In [0]:
df = spark.read.csv("/Volumes/workspace/bronze1/source_files/source_2.csv", header=True, inferSchema=True)
display(df)

In [0]:
%skip
if spark.catalog.tableExists("workspace.bronze1.coffesales"):
    max_ingested = spark.sql("SELECT MAX(`Time`) as max_ingested FROM bronze1.coffesales").collect()[0]['max_ingested']
    if max_ingested is not None:
        df_new = df.filter(col("Time") > max_ingested)
        new_count = df_new.count()
        if new_count > 0:
            print(f"Ingesting {new_count} new records with Time > {max_ingested}")
            df_new.write.format("delta").mode("append").saveAsTable("bronze1.coffesales")
            print("Ingestion completed.")
        else:
            print("No new records to ingest based on Time.")
    else:
        print("No records in target table. Ingesting all source records.")
        df.write.format("delta").mode("append").saveAsTable("bronze1.coffesales")
        print(f"Ingestion completed - {df.count()} records loaded")
else:
    print("Bronze table doesn't exist, creating with initial load")
    df.write.format("delta").mode("overwrite").saveAsTable("bronze1.coffesales")
    print(f"Initial load completed - {df.count()} records loaded")

In [0]:
from pyspark.sql.functions import col, lit

# Align schema: add transaction_id, rename 'time stamp' to 'Time'
df_aligned = df.withColumn("transaction_id", lit(None).cast("long"))
if "time stamp" in df_aligned.columns:
    df_aligned = df_aligned.withColumn("Time", col("time stamp")).drop("time stamp")

if spark.catalog.tableExists("workspace.bronze1.coffesales"):
    max_ingested = spark.sql("SELECT MAX(Time) as max_ingested FROM bronze1.coffesales").collect()[0]['max_ingested']
    if max_ingested is not None:
        df_new = df_aligned.filter(col("Time") > max_ingested)
        new_count = df_new.count()
        if new_count > 0:
            print(f"Ingesting {new_count} new records with Time > {max_ingested}")
            df_new.write.format("delta").mode("append").saveAsTable("bronze1.coffesales")
            print("Ingestion completed.")
        else:
            print("No new records to ingest based on Time.")
    else:
        print("No records in target table. Ingesting all source records.")
        df_aligned.write.format("delta").mode("append").saveAsTable("bronze1.coffesales")
        print(f"Ingestion completed - {df_aligned.count()} records loaded")
else:
    print("Bronze table doesn't exist, creating with initial load")
    df_aligned.write.format("delta").mode("overwrite").saveAsTable("bronze1.coffesales")
    print(f"Initial load completed - {df_aligned.count()} records loaded")